In [1]:
import pandas as pd

df = pd.read_csv("AI_Human.csv")
df.head()

,text,generated
0,Cars. Cars have been around since they became ...,0.0
1,Transportation is a large necessity in most co...,0.0
2,"""America's love affair with it's vehicles seem...",0.0
3,How often do you ride in a car? Do you drive a...,0.0
4,Cars are a wonderful thing. They are perhaps o...,0.0


In [2]:
ai_written=df[df["generated"]== 1].sample(5000, random_state=42)
human_written = df[df["generated"] == 0].sample(5000, random_state=42)

balanced_dataset=pd.concat([ai_written, human_written]).sample(frac=1,random_state=42)

print(balanced_dataset["generated"].value_counts())
print(balanced_dataset.head())

generated
0.0    5000
1.0    5000
Name: count, dtype: int64
                                                     text  generated
269265  Dear Principal,\n\nWe have been hearing quite ...        0.0
327273   Dear [State Senator],\n\nI am writing to expr...        1.0
167398  As high school students, we are constantly bom...        1.0
291678  Hi, I'm a 6th garden and I think zoos ANE Near...        1.0
457139  Sure, jar's my attempt at writing an essay as ...        1.0


In [3]:
balanced_dataset.head()

,text,generated
269265,"Dear Principal,\n\nWe have been hearing quite ...",0.0
327273,"Dear [State Senator],\n\nI am writing to expr...",1.0
167398,"As high school students, we are constantly bom...",1.0
291678,"Hi, I'm a 6th garden and I think zoos ANE Near...",1.0
457139,"Sure, jar's my attempt at writing an essay as ...",1.0


In [4]:
#Train/Test Split

from sklearn.model_selection import train_test_split

X=balanced_dataset["text"]
y=balanced_dataset["generated"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [13]:
#Feature Extraction using text statistics

import numpy as np

def text_extraction(text_statistics):
    new_features=[]
    for text in text_statistics:
        if type(text)== str:
            word=text.split()
            nr_words=len(word)
            nr_chars=len(text)
            unique_words=len(set(word))
            if nr_words:
                avg_word_length=nr_chars/nr_words
            else: 
                avg_word_length=0
            if nr_chars:
                punctuation_ratio= sum(char in".,;:!?'" for char in text) / nr_chars
            else:
                punctuation_ratio=0
            if nr_chars:
                capital_ratio= sum(char.isupper() for char in text) / nr_chars
            else:
                capital_ratio=0
            if nr_chars:
                digit_ratio= sum(char.isdigit() for char in text) / nr_chars
            else:
                digit_ratio=0   
            if nr_words:
                type_token_r=unique_words/nr_words
            else:    
                type_token_r=0

            new_features.append ([nr_words, nr_chars, avg_word_length, punctuation_ratio, capital_ratio, digit_ratio, type_token_r])  

    return np.array(new_features) 

X_train_new_f=text_extraction(X_train)
X_test_new_f=text_extraction(X_test)

In [14]:
# training set
X_train_new_f[:5]

array([[3.38000000e+02, 1.71000000e+03, 5.05917160e+00, 1.34502924e-02,
        8.18713450e-03, 0.00000000e+00, 5.00000000e-01],
       [3.29000000e+02, 1.96400000e+03, 5.96960486e+00, 2.44399185e-02,
        1.32382892e-02, 0.00000000e+00, 5.13677812e-01],
       [2.16000000e+02, 1.20500000e+03, 5.57870370e+00, 8.29875519e-03,
        1.32780083e-02, 4.14937759e-03, 6.06481481e-01],
       [2.35000000e+02, 1.37500000e+03, 5.85106383e+00, 1.60000000e-02,
        2.76363636e-02, 0.00000000e+00, 5.91489362e-01],
       [7.11000000e+02, 4.19100000e+03, 5.89451477e+00, 2.40992603e-02,
        3.41207349e-02, 2.62467192e-03, 4.62728551e-01]])

In [15]:
# training set label
y_train[:5]

459854    0.0
371050    1.0
366180    0.0
281297    1.0
164919    0.0
Name: generated, dtype: float64

In [16]:
# test set
X_test_new_f[:5]

array([[2.54000000e+02, 1.43100000e+03, 5.63385827e+00, 1.74703005e-02,
        1.39762404e-02, 2.09643606e-03, 6.33858268e-01],
       [4.33000000e+02, 2.63800000e+03, 6.09237875e+00, 2.42608036e-02,
        7.58150114e-03, 0.00000000e+00, 4.06466513e-01],
       [3.88000000e+02, 2.36300000e+03, 6.09020619e+00, 1.65044435e-02,
        1.31189166e-02, 0.00000000e+00, 4.20103093e-01],
       [2.79000000e+02, 1.57000000e+03, 5.62724014e+00, 2.99363057e-02,
        3.43949045e-02, 0.00000000e+00, 6.34408602e-01],
       [2.43000000e+02, 1.48700000e+03, 6.11934156e+00, 2.48823134e-02,
        3.09347680e-02, 5.37995965e-03, 6.29629630e-01]])

In [17]:
# test set label
y_test[:5]

86280     0.0
318373    1.0
125370    0.0
140072    1.0
201061    0.0
Name: generated, dtype: float64

## Experiment random forest commit on bofu2

In [56]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,  confusion_matrix

# create random forest model
rf_model = RandomForestClassifier(
    n_estimators=100,    
    max_depth=4,    
    min_samples_split=20,
    min_samples_leaf=20,
    random_state=42
)


rf_model.fit(X_train_new_f, y_train)


y_pred_test = rf_model.predict(X_test_new_f)

# evaluatee
print("test Accuracy:", accuracy_score(y_test, y_pred_test))
print("classification report:", classification_report(y_test, y_pred_test))
print("confusion matrix:", confusion_matrix(y_test, y_pred_test))

test Accuracy: 0.849
classification report:               precision    recall  f1-score   support

         0.0       0.82      0.89      0.85      1000
         1.0       0.88      0.81      0.84      1000

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.85      0.85      0.85      2000

confusion matrix: [[888 112]
 [190 810]]
